# Spin up RDS Multi-AZ, force a failover, and prove a read replica is not a failover target

A senior architect on the team filed a JIRA ticket claiming a read replica gives the same HA guarantees as Multi-AZ and would cut RDS costs in half. The SRE lead disagrees. The CTO asks for a definitive answer with evidence. You have one session to stand up both architectures in the same AWS account, force a failover on the Multi-AZ instance, and prove the read replica does not silently promote to take its place.

You will build:

- **Architecture A: Multi-AZ db.t4g.micro MySQL instance.** Synchronous standby in a second AZ. Failover is automatic and the writer endpoint name does not change.
- **Architecture B: Read replica of Architecture A.** Asynchronous replica with its own endpoint. Promotion is a manual operator action.

Then you call `reboot_db_instance(ForceFailover=True)` on the primary and watch the AvailabilityZone flip while the writer endpoint stays stable. You confirm the read replica remained a read replica and did not silently take over.

**Time.** Plan for about 65 minutes hands-on, mostly waiting on RDS provisioning. The session window is 120 minutes; a first-time Multi-AZ create is 5-10 minutes, the replica create is another 3-5, the failover itself is 60-120 seconds.

**Cost.** Free for AWS accounts within the 12-month RDS free tier (750 hours/month covers a 65-minute session even with Multi-AZ counted as 2 hours/wall-clock hour). Paid accounts spend about 5 to 10 cents end to end. Setup checks free-tier eligibility and surfaces the paid path explicitly.

This lab maps to AWS SAA-C03 Domain 2 (Design Resilient Architectures, 26%).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned per LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import datetime as dt
import getpass
import time

import boto3
from botocore.exceptions import ClientError
from IPython.display import clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "multi-az-rds-vs-read-replica"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"
ENGINE_VERSION = "8.0"  # MySQL 8.0 major; minor resolved by AWS at create time

PRIMARY_ID = f"labrun-{LAB_ID}-primary"
REPLICA_ID = f"labrun-{LAB_ID}-replica"
DB_SUBNET_GROUP_NAME = f"labrun-{LAB_ID}-subnet-group"
SG_NAME = f"labrun-{LAB_ID}-sg"
DB_USERNAME = "labrun_admin"
# Strong throwaway password; the lab does not connect to the database from
# the notebook, only via control-plane RDS APIs.
DB_PASSWORD = "labrun-throwaway-Pw-2026!"

LAB_STATE = {
    "session_start": None,
    "vpc_id": None,
    "sg_id": None,
    "subnet_group_created": False,
    "primary_pre_failover_az": None,
    "primary_pre_failover_endpoint": None,
    "primary_post_failover_az": None,
    "failover_elapsed_seconds": None,
}

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials, and surface free-tier
# eligibility so the student knows whether this session is free or paid.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

# RDS free-tier eligibility note. There is no canonical API for free-tier
# remaining; this prints a directional message so the student knows what
# they are spending. The cost numbers come from the lab brief.
print()
print("RDS billing for this lab:")
print("  Free tier accounts (first 12 months): up to 750 db.t4g.micro hours/month")
print("  free, with Multi-AZ counting as 2 hours per wall-clock hour and read")
print("  replicas counting as 1. A 65-minute lab session consumes about 3 hours.")
print("  Paid accounts: Multi-AZ at $0.034/hour + replica at $0.017/hour = ~$0.05/lab.")
print()

LAB_STATE["session_start"] = dt.datetime.now(dt.timezone.utc)
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# RDS instances are both critical (hourly billed). The rds_db_instance
# adapter calls SkipFinalSnapshot=True and waits for the deleted state
# before returning so the subnet group delete (next in the manifest)
# does not fail with InvalidDBSubnetGroupStateFault.

CLEANUP_MANIFEST = []


def _rebuild_manifest():
    CLEANUP_MANIFEST.clear()
    # Replica first (delete-primary fails if a replica references it).
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="rds_db_instance",
            id=REPLICA_ID,
            region=REGION,
            cli_delete_command=(
                f"aws rds delete-db-instance --db-instance-identifier {REPLICA_ID} "
                f"--skip-final-snapshot --delete-automated-backups"
            ),
        )
    )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="rds_db_instance",
            id=PRIMARY_ID,
            region=REGION,
            cli_delete_command=(
                f"aws rds delete-db-instance --db-instance-identifier {PRIMARY_ID} "
                f"--skip-final-snapshot --delete-automated-backups"
            ),
        )
    )
    if LAB_STATE["subnet_group_created"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="rds_db_subnet_group",
                id=DB_SUBNET_GROUP_NAME,
                region=REGION,
                cli_delete_command=(
                    f"aws rds delete-db-subnet-group "
                    f"--db-subnet-group-name {DB_SUBNET_GROUP_NAME}"
                ),
            )
        )
    if LAB_STATE["sg_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="ec2_security_group",
                id=LAB_STATE["sg_id"],
                region=REGION,
                cli_delete_command=f"aws ec2 delete-security-group --group-id {LAB_STATE['sg_id']}",
            )
        )


_rebuild_manifest()


def _atexit_cleanup():
    try:
        _rebuild_manifest()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to provision RDS resources.")

## Task 1: Provision the Multi-AZ db.t4g.micro MySQL primary (Architecture A)

Three boto3 calls do the work:

1. Build a DB subnet group spanning three AZs from the default VPC. RDS requires at least two AZs for any Multi-AZ instance; three is headroom.
2. Build a security group in the default VPC with no inbound rules (the lab does not connect to MySQL from outside AWS) and the default all-outbound rule.
3. `rds.create_db_instance(...)` with `MultiAZ=True`, `DBInstanceClass=db.t4g.micro`, `Engine=mysql`, `EngineVersion=8.0`. Tag with the lab slug at creation.

The observe cell polls every 30 seconds until `DBInstanceStatus=available`. First-time Multi-AZ creates take 5-10 minutes; the cell prints a status table on each tick so the wait is visible.

In [ ]:
# NBVAL_SKIP
# Task 1: subnet group + security group + Multi-AZ primary.

ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
rds = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Find the default VPC and at least three subnets across distinct AZs.
default_vpcs = ec2.describe_vpcs(
    Filters=[{"Name": "isDefault", "Values": ["true"]}],
)["Vpcs"]
if not default_vpcs:
    print("No default VPC in this account. Lab 5 expects a default VPC.")
    raise SystemExit(1)
LAB_STATE["vpc_id"] = default_vpcs[0]["VpcId"]
print(f"Default VPC: {LAB_STATE['vpc_id']}")

subnets = ec2.describe_subnets(
    Filters=[{"Name": "vpc-id", "Values": [LAB_STATE["vpc_id"]]}],
)["Subnets"]
subnets_by_az = {}
for sn in subnets:
    subnets_by_az.setdefault(sn["AvailabilityZone"], sn["SubnetId"])
selected_az = sorted(subnets_by_az.keys())[:3]
selected_subnets = [subnets_by_az[az] for az in selected_az]
if len(selected_subnets) < 2:
    print("Default VPC must have subnets in at least 2 AZs for Multi-AZ RDS.")
    raise SystemExit(1)
print(f"Subnet group AZs: {selected_az}")

# Create the DB subnet group.
# YOUR CODE: call rds.create_db_subnet_group with DBSubnetGroupName=DB_SUBNET_GROUP_NAME,
# DBSubnetGroupDescription="labrun multi-az lab subnet group",
# SubnetIds=selected_subnets, and Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Handle the DBSubnetGroupAlreadyExists error by treating it as a no-op.

LAB_STATE["subnet_group_created"] = True

# Create the security group with no inbound rules.
try:
    sg = ec2.create_security_group(
        GroupName=SG_NAME,
        Description="labrun multi-az lab security group",
        VpcId=LAB_STATE["vpc_id"],
        TagSpecifications=[
            {
                "ResourceType": "security-group",
                "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
            }
        ],
    )
    LAB_STATE["sg_id"] = sg["GroupId"]
except ClientError as e:
    if e.response.get("Error", {}).get("Code") == "InvalidGroup.Duplicate":
        sgs = ec2.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [SG_NAME]}],
        )["SecurityGroups"]
        LAB_STATE["sg_id"] = sgs[0]["GroupId"]
    else:
        raise

_rebuild_manifest()
print(f"Security group: {LAB_STATE['sg_id']}")

# Provision the Multi-AZ primary.
# YOUR CODE: call rds.create_db_instance with DBInstanceIdentifier=PRIMARY_ID,
# Engine="mysql", EngineVersion=ENGINE_VERSION, DBInstanceClass="db.t4g.micro",
# AllocatedStorage=20, MasterUsername=DB_USERNAME, MasterUserPassword=DB_PASSWORD,
# MultiAZ=True, DBSubnetGroupName=DB_SUBNET_GROUP_NAME,
# VpcSecurityGroupIds=[LAB_STATE["sg_id"]], PubliclyAccessible=False,
# BackupRetentionPeriod=0, DeletionProtection=False, StorageEncrypted=False,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Handle DBInstanceAlreadyExistsFault as a no-op.

print(f"Multi-AZ primary requested: {PRIMARY_ID}")
print("Now run the observe cell to watch it reach available (5-10 minutes).")

In [ ]:
# NBVAL_SKIP
# Observe: poll the primary RDS instance until DBInstanceStatus=available.
# Ceiling 20 minutes (Multi-AZ provisioning is the longest single AWS API
# wait in the SAA track).

rds_obs = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 1200
status = "creating"
while time.time() < deadline:
    clear_output(wait=True)
    try:
        inst = rds_obs.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
        status = inst.get("DBInstanceStatus")
        az = inst.get("AvailabilityZone")
        secondary_az = inst.get("SecondaryAvailabilityZone")
        multi_az = inst.get("MultiAZ")
        endpoint = (inst.get("Endpoint") or {}).get("Address", "pending")
    except ClientError as e:
        status = f"error: {e}"
        az = secondary_az = endpoint = "?"
        multi_az = "?"
    elapsed = int(1200 - (deadline - time.time()))
    print(f"Multi-AZ primary status at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  Identifier:       {PRIMARY_ID}")
    print(f"  Status:           {status}")
    print(f"  MultiAZ:          {multi_az}")
    print(f"  Primary AZ:       {az}")
    print(f"  Secondary AZ:     {secondary_az}")
    print(f"  Writer endpoint:  {endpoint}")
    if status == "available" and multi_az:
        LAB_STATE["primary_pre_failover_az"] = az
        LAB_STATE["primary_pre_failover_endpoint"] = endpoint
        print()
        print("Multi-AZ primary is available. AZ captured for Checkpoint 3.")
        break
    time.sleep(30)
else:
    print()
    print("Timed out before the primary reached available. Try again.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Multi-AZ primary exists, available, MultiAZ=True, AZ captured.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        rds_c = boto3.client(
            "rds",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        inst = rds_c.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
        if inst.get("DBInstanceClass") != "db.t4g.micro":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Primary class is {inst.get('DBInstanceClass')!r}, expected db.t4g.micro."
                ),
            )
        if inst.get("Engine") != "mysql":
            return CheckpointResult(
                status="fail",
                error_reason=f"Primary engine is {inst.get('Engine')!r}, expected mysql.",
            )
        if not inst.get("MultiAZ"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Primary MultiAZ=False. The lab pivots on Multi-AZ failover; "
                    "recreate the instance with MultiAZ=True."
                ),
            )
        if inst.get("DBInstanceStatus") != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Primary status is {inst.get('DBInstanceStatus')!r}, expected available."
                ),
            )

        arn = inst.get("DBInstanceArn")
        try:
            tags_resp = rds_c.list_tags_for_resource(ResourceName=arn)
            tags = {t["Key"]: t["Value"] for t in tags_resp.get("TagList", [])}
        except ClientError:
            tags = {}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Primary is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found: {tags}"
                ),
            )

        if not LAB_STATE["primary_pre_failover_az"]:
            LAB_STATE["primary_pre_failover_az"] = inst.get("AvailabilityZone")
            LAB_STATE["primary_pre_failover_endpoint"] = (
                inst.get("Endpoint") or {}
            ).get("Address")

        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Two calls. `create_db_subnet_group` with the subnet IDs from the default VPC, then `create_db_instance` with `MultiAZ=True`. Tag both with the lab slug so the orphan scan and tag audit find them.

</details>

<details><summary>Hint 2 (direction)</summary>

`create_db_instance` is asynchronous and returns immediately. Status moves from `creating` to `backing-up` to `available` over 5-10 minutes. The observe cell handles the wait; you do not need a waiter call yourself. The arguments that matter: `MultiAZ=True`, `DBSubnetGroupName=DB_SUBNET_GROUP_NAME`, `VpcSecurityGroupIds=[sg_id]`, `BackupRetentionPeriod=0` (Multi-AZ requires backups enabled but 0 disables them; if AWS rejects 0 set it to 1).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
try:
    rds.create_db_subnet_group(
        DBSubnetGroupName=DB_SUBNET_GROUP_NAME,
        DBSubnetGroupDescription="labrun multi-az lab subnet group",
        SubnetIds=selected_subnets,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "DBSubnetGroupAlreadyExists":
        raise

try:
    rds.create_db_instance(
        DBInstanceIdentifier=PRIMARY_ID,
        Engine="mysql",
        EngineVersion=ENGINE_VERSION,
        DBInstanceClass="db.t4g.micro",
        AllocatedStorage=20,
        MasterUsername=DB_USERNAME,
        MasterUserPassword=DB_PASSWORD,
        MultiAZ=True,
        DBSubnetGroupName=DB_SUBNET_GROUP_NAME,
        VpcSecurityGroupIds=[LAB_STATE["sg_id"]],
        PubliclyAccessible=False,
        BackupRetentionPeriod=1,
        DeletionProtection=False,
        StorageEncrypted=False,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "DBInstanceAlreadyExists":
        raise
```

</details>

**Wallet check.** Free-tier accounts: 0 cents. Paid accounts: about 0.6 cents per minute since the primary started ($0.034/hour for Multi-AZ). The subnet group and security group are free.

## Task 2: Create a read replica of the primary (Architecture B)

`create_db_instance_read_replica` is the single call. The replica:

- Is single-AZ by default; the lab does not enable Multi-AZ on the replica (multi-AZ on a replica is a separate paid feature outside the lab's scope).
- Inherits the engine version and parameter group from the source.
- Gets its own DB instance identifier (`REPLICA_ID`).
- Carries `ReadReplicaSourceDBInstanceIdentifier` pointing back at the primary.

The observe cell polls until the replica reaches `available`. Replicas come up faster than fresh primaries (3-5 minutes typical) because RDS clones from the source's existing snapshot.

In [ ]:
# NBVAL_SKIP
# Task 2: create the read replica.

rds = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: call rds.create_db_instance_read_replica with
# DBInstanceIdentifier=REPLICA_ID, SourceDBInstanceIdentifier=PRIMARY_ID,
# DBInstanceClass="db.t4g.micro", PubliclyAccessible=False,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Handle DBInstanceAlreadyExists as a no-op.

print(f"Read replica requested: {REPLICA_ID}")
print("Run the observe cell to watch it reach available (3-5 minutes).")

In [ ]:
# NBVAL_SKIP
# Observe: poll the read replica until DBInstanceStatus=available.

rds_obs = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 600
while time.time() < deadline:
    clear_output(wait=True)
    try:
        inst = rds_obs.describe_db_instances(DBInstanceIdentifier=REPLICA_ID)["DBInstances"][0]
        status = inst.get("DBInstanceStatus")
        source = inst.get("ReadReplicaSourceDBInstanceIdentifier")
        az = inst.get("AvailabilityZone")
    except ClientError as e:
        status = f"error: {e}"
        source = az = "?"
    elapsed = int(600 - (deadline - time.time()))
    print(f"Read replica status at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  Identifier: {REPLICA_ID}")
    print(f"  Status:     {status}")
    print(f"  Source:     {source}")
    print(f"  AZ:         {az}")
    if status == "available":
        print()
        print("Replica is available and pointed at the primary.")
        break
    time.sleep(30)
else:
    print()
    print("Timed out before the replica reached available.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: replica exists, available, references the primary, MultiAZ=False.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        rds_c = boto3.client(
            "rds",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        inst = rds_c.describe_db_instances(DBInstanceIdentifier=REPLICA_ID)["DBInstances"][0]
        if inst.get("DBInstanceStatus") != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Replica status is {inst.get('DBInstanceStatus')!r}, expected available."
                ),
            )
        source = inst.get("ReadReplicaSourceDBInstanceIdentifier", "")
        if PRIMARY_ID not in source:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Replica source is {source!r}; expected to reference {PRIMARY_ID}."
                ),
            )
        if inst.get("MultiAZ"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Replica MultiAZ=True. The lab uses a single-AZ replica to isolate "
                    "the comparison; recreate without --multi-az."
                ),
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

A read replica is a separate database instance pointed at the source via configuration. AWS provides a dedicated API call that handles the replication wiring for you; you do not run a `create_db_instance` plus a manual replication setup.

</details>

<details><summary>Hint 2 (direction)</summary>

`rds.create_db_instance_read_replica` takes `DBInstanceIdentifier` for the new replica and `SourceDBInstanceIdentifier` for the primary. Class can be any compatible size; the lab pins `db.t4g.micro` so both instances are the same size and the comparison is apples to apples.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
try:
    rds.create_db_instance_read_replica(
        DBInstanceIdentifier=REPLICA_ID,
        SourceDBInstanceIdentifier=PRIMARY_ID,
        DBInstanceClass="db.t4g.micro",
        PubliclyAccessible=False,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "DBInstanceAlreadyExists":
        raise
```

</details>

**Wallet check.** Free tier still 0 cents. Paid accounts: replica is $0.017/hour, primary Multi-AZ is $0.034/hour. Combined burn is about 0.085 cents per minute.

## Task 3: Force a failover on the Multi-AZ primary

The actual test. `reboot_db_instance(DBInstanceIdentifier=PRIMARY_ID, ForceFailover=True)` does three things in the AWS control plane:

1. AWS marks the current primary as failing.
2. The synchronous standby in the secondary AZ is promoted to primary in 60-120 seconds.
3. The writer endpoint's DNS A record is repointed at the new primary IP.

You capture two side effects:

- `AvailabilityZone` flips to what was previously `SecondaryAvailabilityZone`.
- The endpoint hostname (e.g., `labrun-multi-az-rds-vs-read-replica-primary.cluster-xxx.us-east-1.rds.amazonaws.com`) does NOT change. The application code keeps writing to the same endpoint name; the DNS layer absorbed the failover.

The observe cell polls `describe_db_instances` every 10 seconds for up to 4 minutes. It prints the AZ on each tick so the flip is visible in real time.

In [ ]:
# NBVAL_SKIP
# Task 3: trigger the failover. Most of the work is in the observe cell;
# this cell just fires reboot_db_instance.

rds = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Capture pre-failover state.
inst = rds.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
LAB_STATE["primary_pre_failover_az"] = inst.get("AvailabilityZone")
LAB_STATE["primary_pre_failover_endpoint"] = (inst.get("Endpoint") or {}).get("Address")
print(f"Pre-failover AZ:       {LAB_STATE['primary_pre_failover_az']}")
print(f"Pre-failover endpoint: {LAB_STATE['primary_pre_failover_endpoint']}")
print()

# Fire the failover.
# YOUR CODE: call rds.reboot_db_instance(DBInstanceIdentifier=PRIMARY_ID,
# ForceFailover=True). Capture nothing; the observe cell handles the
# wall-clock duration measurement.

print("Failover triggered. Run the observe cell to watch the AZ flip.")

In [ ]:
# NBVAL_SKIP
# Observe: poll until DBInstanceStatus returns to available AND
# AvailabilityZone has flipped. Capture the wall-clock duration so the
# notebook surfaces it. Typical 60-120 seconds; ceiling 4 minutes.

rds_obs = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

start = time.time()
deadline = start + 240
while time.time() < deadline:
    clear_output(wait=True)
    try:
        inst = rds_obs.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
        status = inst.get("DBInstanceStatus")
        az = inst.get("AvailabilityZone")
        secondary = inst.get("SecondaryAvailabilityZone")
        endpoint = (inst.get("Endpoint") or {}).get("Address", "pending")
    except ClientError as e:
        status = f"error: {e}"
        az = secondary = endpoint = "?"
    elapsed = int(time.time() - start)
    print(f"Failover progress at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  Status:           {status}")
    print(f"  AZ now:           {az}")
    print(f"  AZ before:        {LAB_STATE['primary_pre_failover_az']}")
    print(f"  Secondary AZ now: {secondary}")
    print(f"  Endpoint now:     {endpoint}")
    print(f"  Endpoint before:  {LAB_STATE['primary_pre_failover_endpoint']}")
    flipped = (
        status == "available"
        and az
        and az != LAB_STATE["primary_pre_failover_az"]
    )
    if flipped:
        LAB_STATE["primary_post_failover_az"] = az
        LAB_STATE["failover_elapsed_seconds"] = elapsed
        print()
        print(f"Failover complete in {elapsed} seconds.")
        print(f"  Writer endpoint unchanged: {endpoint == LAB_STATE['primary_pre_failover_endpoint']}")
        break
    time.sleep(10)
else:
    print()
    print("Timed out waiting for the AZ flip. Possible causes:")
    print("  - MultiAZ=False on the primary; rerun Checkpoint 1 and recreate.")
    print("  - reboot_db_instance was called without ForceFailover=True.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: ForceFailover flipped the AZ and the writer endpoint stayed.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        rds_c = boto3.client(
            "rds",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        inst = rds_c.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
        post_az = inst.get("AvailabilityZone")
        pre_az = LAB_STATE.get("primary_pre_failover_az")
        post_endpoint = (inst.get("Endpoint") or {}).get("Address")
        pre_endpoint = LAB_STATE.get("primary_pre_failover_endpoint")
        if not pre_az:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Pre-failover AZ was never captured. Run the Task 3 cell that "
                    "fires reboot_db_instance before this checkpoint."
                ),
            )
        if inst.get("DBInstanceStatus") != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Primary is in {inst.get('DBInstanceStatus')!r}; failover may "
                    f"still be in progress."
                ),
            )
        if post_az == pre_az:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AvailabilityZone did not flip ({pre_az} -> {post_az}). "
                    f"Most common cause: ForceFailover=True was not passed to "
                    f"reboot_db_instance."
                ),
            )
        if pre_endpoint and post_endpoint and post_endpoint != pre_endpoint:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Writer endpoint changed across failover "
                    f"({pre_endpoint} -> {post_endpoint}). The whole point of "
                    f"Multi-AZ is endpoint stability."
                ),
            )
        if not LAB_STATE.get("failover_elapsed_seconds"):
            LAB_STATE["failover_elapsed_seconds"] = 0
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

One boto3 call triggers the failover. The same API is used to reboot an instance normally; a single boolean parameter changes the semantics.

</details>

<details><summary>Hint 2 (direction)</summary>

`reboot_db_instance(DBInstanceIdentifier=..., ForceFailover=True)`. Without `ForceFailover=True` the reboot is in-place and the AZ does not change. Capture the pre-failover AZ before the call (already done for you) and let the observe cell handle the polling.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
rds.reboot_db_instance(DBInstanceIdentifier=PRIMARY_ID, ForceFailover=True)
```

</details>

**Wallet check.** The failover itself is a no-cost control-plane action; no extra hours, no extra storage. The total session burn is still the same Multi-AZ + replica per-hour rate.

## Task 4: Prove the read replica is still a read replica

This is the SRE lead's argument made concrete. After Architecture A's failover, Architecture B's replica should:

- Still report `DBInstanceStatus=available`.
- Still carry `ReadReplicaSourceDBInstanceIdentifier` pointing back at the primary.
- Still appear in the primary's `ReadReplicaDBInstanceIdentifiers` list.

If the architect's claim were correct, the replica would have been silently promoted to take over when the primary failed. AWS does not do that. A read replica is a read replica until an operator explicitly calls `promote_read_replica`.

This task is a verification cell; you do not create anything new. The observe cell prints the replica's role state alongside the primary's, side by side, so the SRE lead's point is visible in the notebook output.

In [ ]:
# NBVAL_SKIP
# Task 4: verify the replica's role did not change. No new resources created.

rds = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Read both instances' state. Print a side-by-side comparison so the
# pedagogical point is visible.
# YOUR CODE: call rds.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)
# and rds.describe_db_instances(DBInstanceIdentifier=REPLICA_ID).
# Print a table showing for each: DBInstanceIdentifier, role (primary if
# ReadReplicaSourceDBInstanceIdentifier is empty else "read replica"),
# DBInstanceStatus, AvailabilityZone, and (for the primary) the
# ReadReplicaDBInstanceIdentifiers list.

In [ ]:
# NBVAL_SKIP
# Observe: print the role comparison and confirm the replica continues to
# reference the primary post-failover. Single snapshot, not a poll loop.

rds_obs = boto3.client(
    "rds",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

clear_output(wait=True)
try:
    primary = rds_obs.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
    replica = rds_obs.describe_db_instances(DBInstanceIdentifier=REPLICA_ID)["DBInstances"][0]
except ClientError as e:
    print(f"Error reading instance state: {e}")
    primary = replica = {}

print("Architecture A vs Architecture B after the failover:")
print("=" * 78)
print(f"  {'attribute':30}  {'Architecture A':22}  {'Architecture B':22}")
print("-" * 78)
print(
    f"  {'DBInstanceIdentifier':30}  "
    f"{primary.get('DBInstanceIdentifier', '?'):22}  "
    f"{replica.get('DBInstanceIdentifier', '?'):22}"
)
role_a = (
    "read replica" if primary.get("ReadReplicaSourceDBInstanceIdentifier") else "primary (writer)"
)
role_b = (
    "read replica" if replica.get("ReadReplicaSourceDBInstanceIdentifier") else "primary (writer)"
)
print(f"  {'Role':30}  {role_a:22}  {role_b:22}")
print(
    f"  {'DBInstanceStatus':30}  "
    f"{primary.get('DBInstanceStatus', '?'):22}  "
    f"{replica.get('DBInstanceStatus', '?'):22}"
)
print(
    f"  {'AvailabilityZone':30}  "
    f"{primary.get('AvailabilityZone', '?'):22}  "
    f"{replica.get('AvailabilityZone', '?'):22}"
)
print(
    f"  {'MultiAZ':30}  "
    f"{str(primary.get('MultiAZ', '?')):22}  "
    f"{str(replica.get('MultiAZ', '?')):22}"
)
print(
    f"  {'Source ref (replica only)':30}  "
    f"{'-':22}  "
    f"{replica.get('ReadReplicaSourceDBInstanceIdentifier', '?'):22}"
)
known_replicas = primary.get("ReadReplicaDBInstanceIdentifiers", [])
print(f"  {'Primary lists replicas':30}  {str(known_replicas):22}  {'-':22}")
print()
print("Architecture A failed over via automatic standby promotion; endpoint stable.")
print("Architecture B remained a read replica throughout. AWS did NOT promote it.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: replica did not promote across the failover.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        rds_c = boto3.client(
            "rds",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        replica = rds_c.describe_db_instances(DBInstanceIdentifier=REPLICA_ID)["DBInstances"][0]
        source = replica.get("ReadReplicaSourceDBInstanceIdentifier", "")
        if not source or PRIMARY_ID not in source:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Replica no longer references the primary (got {source!r}). "
                    f"Either it was promoted manually or it is the wrong instance."
                ),
            )
        if replica.get("DBInstanceStatus") not in ("available", "modifying", "backing-up"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Replica status is {replica.get('DBInstanceStatus')!r}; "
                    f"expected available or transitional."
                ),
            )

        primary = rds_c.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
        known_replicas = primary.get("ReadReplicaDBInstanceIdentifiers", [])
        if REPLICA_ID not in known_replicas:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Primary {PRIMARY_ID} does not list {REPLICA_ID} in its "
                    f"ReadReplicaDBInstanceIdentifiers. The replica may have been "
                    f"promoted (which deletes the replica relationship)."
                ),
            )

        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

This task does not require a new boto3 call; you re-read the same instances and inspect their relationship. The comparison table in the observe cell tells you what to print.

</details>

<details><summary>Hint 2 (direction)</summary>

`rds.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)` and the same for the replica. Look at three fields per instance: `DBInstanceIdentifier`, `ReadReplicaSourceDBInstanceIdentifier` (empty on the primary, set on the replica), and `DBInstanceStatus`. The point is that the replica's source has NOT changed across the failover.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
primary = rds.describe_db_instances(DBInstanceIdentifier=PRIMARY_ID)["DBInstances"][0]
replica = rds.describe_db_instances(DBInstanceIdentifier=REPLICA_ID)["DBInstances"][0]
print("Primary:", primary["DBInstanceIdentifier"], primary["DBInstanceStatus"], primary["AvailabilityZone"])
print("Replica:", replica["DBInstanceIdentifier"], replica["DBInstanceStatus"], replica["AvailabilityZone"])
print("Replica source:", replica.get("ReadReplicaSourceDBInstanceIdentifier"))
print("Primary's replicas:", primary.get("ReadReplicaDBInstanceIdentifiers"))
```

</details>

**Wallet check.** Same per-hour burn. The verification cell makes one extra `describe_db_instances` call (free). Running total: about 5 to 10 cents on a paid account, 0 cents on the free tier.

## Cleanup

Both RDS instances are critical (hourly billed). The cleanup adapter deletes the replica first (deleting the primary while a replica references it fails), waits for the replica to reach `deleted`, then deletes the primary with `SkipFinalSnapshot=True`. Total cleanup wall-clock is 8-12 minutes; do not close the notebook tab until the cleanup card lands.

In [ ]:
# NBVAL_SKIP
# Cleanup. RDS delete is asynchronous; the rds_db_instance adapter waits
# for the deleted state before returning so the subnet group can be
# deleted without InvalidDBSubnetGroupStateFault.

import sys

_rebuild_manifest()
result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "rds_db_instance")
critical_destroyed = sum(
    1
    for r in CLEANUP_MANIFEST
    if r.type == "rds_db_instance" and r.id not in failed_ids
)
standard_total = len(CLEANUP_MANIFEST) - critical_total
standard_destroyed = standard_total - sum(
    1
    for rid in failed_ids
    for r in CLEANUP_MANIFEST
    if r.id == rid and r.type != "rds_db_instance"
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00 on free tier, $0.05 to $0.10 paid.** RDS storage is prorated for the session length so storage cost is a few cents; instance hours are the dominant line item. The cleanup cell tore both instances down with `SkipFinalSnapshot=True` so there is no residual snapshot cost.

## Reflection

These are not graded. They are for you.

1. Walk through what AWS does in the first 60 seconds after a Multi-AZ RDS primary fails. List the steps: failure detection, DNS endpoint flip, standby promotion, client reconnect. Where does the writer endpoint name fit in that flow, and why does the application code not need to change to keep writing?

2. Your team is choosing between Multi-AZ (one primary, one synchronous standby, automatic failover, no read scaling) and a single-AZ primary with three read replicas (asynchronous replication, no automatic failover, 3x read scaling). For a workload with 90% reads and a strict 60-second RTO, which architecture fits, and what would you add to the loser to compensate?

## Exam-Style Practice

**Question 1 (Domain 2, Multi-AZ vs read replica failover semantics):**

A team runs a production OLTP database on RDS. The application has a strict 60-second RTO on the writer endpoint and the team wants AWS-managed automatic failover with no application code changes. Which option fits?

A. Single-AZ RDS primary with two read replicas in different AZs. On primary failure, the team manually promotes a replica via `promote_read_replica`.

B. Multi-AZ RDS instance with synchronous standby in a second AZ. AWS handles failover automatically via DNS endpoint flip.

C. Single-AZ RDS primary with daily snapshots and a documented runbook for restoring from snapshot into a new instance.

D. Two single-AZ RDS primaries in different AZs with the application writing to both manually.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: read replicas are NOT automatically promoted on primary failure. Manual promotion via `promote_read_replica` is an operator action that takes minutes, not seconds. RTO unmet.
- B is correct: Multi-AZ RDS maintains a synchronous standby in a second AZ. On primary failure AWS detects within 10-30 seconds, flips the writer endpoint's DNS to the standby, and the application reconnects to the same endpoint name. Total failover wall-clock is typically 60-120 seconds, within the RTO.
- C is wrong: snapshot restore takes 15-60 minutes depending on database size. RTO of 60 seconds is unmet by an order of magnitude.
- D is wrong: writing to two primaries manually is not RDS HA; it is application-managed multi-primary which is not supported by RDS for MySQL or Postgres and introduces split-brain risk.

</details>

**Question 2 (Domain 2, when read replicas are the right tool):**

A workload is 95% reads and 5% writes. The writes are tolerant of eventual consistency on the read side. The team wants to scale read throughput 3x without changing the application's connection string for writes. Which architecture fits?

A. Multi-AZ RDS instance; the standby absorbs read traffic automatically.

B. Multi-AZ RDS instance plus three read replicas in different AZs; the application reads from a replica endpoint pool and writes to the writer endpoint.

C. Three single-AZ RDS instances with application-side sharding on tenant ID.

D. Aurora Serverless v2 with auto-scaling Aurora Capacity Units.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: Multi-AZ RDS standby does NOT serve read traffic. The standby is a passive replica that exists for failover only. Reading from it is not supported on standard RDS.
- B is correct: Multi-AZ on the primary gives automatic failover. Read replicas scale read throughput independently and serve reads with eventual consistency. The application reads from a replica-endpoint pool (with client-side load balancing) and writes to the unchanged writer endpoint. This is the AWS-recommended pattern for read-heavy workloads.
- C is wrong: sharding adds application-level complexity (cross-shard queries, rebalancing) that the question does not require. Read replicas solve the read-scaling problem without sharding.
- D is partially right that Aurora Serverless can autoscale, but it changes the database engine and connection model. The question asks about scaling within RDS without changing connection strings.

</details>